# 🔮 KuroNeko TR v2 - Phi-4 Mini Türkçe Eğitim (100K + Derin Düşünme)

**Platform:** Kaggle GPU (H100/T4, Internet Var)  
**Model:** microsoft/Phi-4-mini-instruct (3.8B)  
**Yöntem:** QLoRA (4-bit)  
**Veri:** 100K+ Türkçe örnek  
**Epoch:** 3  

In [ ]:
# ── 1. KURULUM ──────────────────────────────────────────────────────────────
import subprocess, sys, os

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-500:])
    print(result.stdout[-300:])

print('📦 Paketler kuruluyor...')
run('pip install -q --upgrade pip')
run('pip install -q --upgrade huggingface_hub transformers peft bitsandbytes accelerate datasets trl')
print('✅ Kurulum tamamlandı')

In [ ]:
# ── 2. HF TOKEN ─────────────────────────────────────────────────────────────
import os
HF_TOKEN = ''  #@param {type:'string'}
os.environ['HF_TOKEN'] = HF_TOKEN
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('✅ HF login OK')
else:
    print('⚠️ HF_TOKEN boş - push atlanacak')

In [ ]:
# ── 3. GPU KONTROL ──────────────────────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU sayısı: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory/1e9:.1f} GB)')

In [ ]:
# ── 4. VERİ SETİ HAZIRLA (100K) ────────────────────────────────────────────
from datasets import load_dataset, Dataset
import random, json, os

random.seed(42)
os.makedirs('/kaggle/working/data', exist_ok=True)

all_data = []

def add_sample(q, a):
    q = str(q).strip()
    a = str(a).strip()
    if q and a and len(q) > 5 and len(a) > 5:
        all_data.append({'q': q[:800], 'a': a[:800]})

def safe_load(ds_name, col_q, col_a, col_ctx=None, limit=15000):
    try:
        ds = load_dataset(ds_name, split='train', cache_dir='/kaggle/working/hf_cache')
        if len(ds) > limit:
            ds = ds.shuffle(seed=42).select(range(limit))
        cnt = 0
        for row in ds:
            q = str(row.get(col_q, '')).strip()
            a = str(row.get(col_a, '')).strip()
            if col_ctx and row.get(col_ctx):
                ctx = str(row[col_ctx]).strip()
                if ctx:
                    q = ctx + ' ' + q
            add_sample(q, a)
            cnt += 1
        print(f'  ✅ {ds_name}: {cnt}')
    except Exception as e:
        print(f'  ⚠️ {ds_name}: {str(e)[:80]}')

print('📥 Phase 1: HF veri setleri...')
safe_load('merve/turkish_instructions', 'instruction', 'output', 'input', 15000)
safe_load('ucekmez/OpenOrca-tr', 'question', 'response', None, 15000)
safe_load('atasoglu/databricks-dolly-15k-tr', 'instruction', 'response', 'context', 15000)
safe_load('ytu-ce-cosmos/gsm8k_tr', 'question', 'answer', None, 8000)
safe_load('beratcmn/no_robots_turkish', 'instruction', 'output', None, 8000)
safe_load('beratcmn/lima-tr', 'instruction', 'output', None, 5000)
safe_load('emre/stanford-alpaca-cleaned-turkish-translated', 'instruction', 'output', 'input', 10000)
safe_load('alibayram/turkish_mmlu', 'question', 'answer', None, 8000)
safe_load('nisancoskun/turkish_general_knowledge_qa', 'question', 'answer', None, 8000)
print(f'HF toplam: {len(all_data)}')

In [ ]:
# ── 4B. SENTETİK VERİ (30K+) ────────────────────────────────────────────────
import random as rnd
rnd.seed(42)

print('🔧 Phase 2: Sentetik veri...')

# Matematik CoT (3000)
for _ in range(500):
    a,b,c,d = rnd.randint(3,50), rnd.randint(3,50), rnd.randint(1,10), rnd.randint(1,10)
    q = str(a) + ' TL/kg elma, ' + str(b) + ' TL/kg armut. ' + str(c) + ' kg elma + ' + str(d) + ' kg armut = ? Adım adım çöz.'
    ans = 'Adım adım:\nElma: ' + str(a) + '×' + str(c) + '=' + str(a*c) + ' TL\nArmut: ' + str(b) + '×' + str(d) + '=' + str(b*d) + ' TL\nToplam: ' + str(a*c+b*d) + ' TL'
    add_sample(q, ans)

for _ in range(500):
    u,k = rnd.randint(5,50), rnd.randint(3,40)
    q = 'Dikdörtgen: uzun=' + str(u) + ' cm, kısa=' + str(k) + ' cm. Alan ve çevre? Adım adım.'
    ans = 'Adım adım:\nAlan: ' + str(u) + '×' + str(k) + '=' + str(u*k) + ' cm²\nÇevre: 2×(' + str(u) + '+' + str(k) + ')=' + str(2*(u+k)) + ' cm'
    add_sample(q, ans)

for _ in range(500):
    t = rnd.randint(50,500)
    p = rnd.choice([5,10,15,20,25,30,40,50])
    r = int(t*p/100)
    q = str(t) + ' sayısının %' + str(p) + 'si kaçtır? Adım adım.'
    ans = 'Adım adım:\n' + str(t) + '×' + str(p) + '/100=' + str(r) + '\nSonuç: ' + str(r)
    add_sample(q, ans)

for _ in range(500):
    v,h = rnd.randint(40,120), rnd.randint(1,12)
    q = str(v) + ' km/h ile ' + str(h) + ' saat kaç km? Adım adım.'
    ans = 'Adım adım:\nMesafe=' + str(v) + '×' + str(h) + '=' + str(v*h) + ' km'
    add_sample(q, ans)

for _ in range(500):
    nums = [rnd.randint(1,100) for _ in range(rnd.randint(3,7))]
    avg = sum(nums)/len(nums)
    q = ', '.join(map(str, nums)) + ' ortalaması? Adım adım.'
    ans = 'Adım adım:\nToplam=' + str(sum(nums)) + ', Adet=' + str(len(nums)) + '\nOrtalama=' + str(sum(nums)) + '/' + str(len(nums)) + '=' + str(round(avg,1))
    add_sample(q, ans)

for _ in range(500):
    s,st = rnd.randint(1,20), rnd.randint(2,7)
    seq = [s+st*i for i in range(8)]
    q = 'Sonraki sayı: ' + ', '.join(map(str, seq[:6])) + ', ? Adım adım.'
    ans = 'Adım adım:\nFark=' + str(st) + ', Sonraki=' + str(seq[5]) + '+' + str(st) + '=' + str(seq[6])
    add_sample(q, ans)

print('  Matematik CoT: 3000')

# Kod CoT (3000)
code_qa = [
    ("Python'da liste nasıl sıralanır? Adım adım açıkla.",
     "Adım adım:\n1. sorted(): yeni liste\n```python\nters=sorted(liste,reverse=True)\n```\n2. sort(): yerinde\n```python\nliste.sort()\n```\n3. Anahtara göre:\n```python\nsorted(ogs,key=lambda x:x['yas'])\n```\nÖnerilen: sorted() Pythonic."),
    ("Python'da dosya nasıl okunur? Adım adım açıkla.",
     "Adım adım:\n1. with (önerilen):\n```python\nwith open('f.txt','r',encoding='utf-8') as f:\n    icerik=f.read()\n```\n2. Satır satır:\n```python\nfor satir in f:\n    print(satir.strip())\n```\n3. Liste:\n```python\nsatirlar=f.readlines()\n```"),
    ("Git commit nasıl atılır? Adım adım açıkla.",
     "Adım adım:\n1. Ekle:\n```bash\ngit add .\n```\n2. Commit:\n```bash\ngit commit -m 'mesaj'\n```\n3. Push:\n```bash\ngit push origin main\n```"),
    ("Docker container arka planda nasıl çalıştırılır? Adım adım.",
     "Adım adım:\n1. Arka plan:\n```bash\ndocker run -d --name konteyner image\n```\n2. Port:\n```bash\ndocker run -d -p 8080:80 nginx\n```\n3. Yönetim:\n```bash\ndocker exec konteyner komut\ndocker stop konteyner\n```"),
    ("SQL injection nedir, nasıl önlenir? Adım adım.",
     "Adım adım:\n1. Tehlike (KÖTÜ):\n```python\ncursor.execute(f'SELECT * FROM users WHERE id={uid}')\n```\n2. Çözüm (İYİ):\n```python\ncursor.execute('SELECT * FROM users WHERE id=%s',(uid,))\n```\n3. En iyi: ORM kullan."),
    ("Python'da API'ye HTTP isteği nasıl atılır? Adım adım.",
     "Adım adım:\n1. GET:\n```python\nimport requests\nr=requests.get('https://api.example.com')\nprint(r.json())\n```\n2. POST:\n```python\nr=requests.post(url,json={'ad':'Ali'})\n```\n3. Hata:\n```python\ntry:\n    r.raise_for_status()\nexcept Exception as e:\n    print(e)\n```"),
    ("Linux'ta dosya izinleri nasıl değiştirilir? Adım adım.",
     "Adım adım:\n1. Sayısal:\n```bash\nchmod 755 dosya.sh\nchmod 644 dosya.txt\n```\n2. Harfli:\n```bash\nchmod +x dosya.sh\nchmod u+w dosya.txt\n```\n3. Değerler: 4=Okuma, 2=Yazma, 1=Çalıştırma"),
]
for q, a in code_qa * 428:
    add_sample(q, a)
print('  Kod CoT: ~3000')

# Akademik CoT (10000)
aca_qa = [
    ("Fotosentez nedir? Adım adım açıkla.",
     "Adım adım fotosentez:\n1. Tanım: Bitkiler güneş ışığını kimyasal enerjiye çevirir.\n2. Yer: Kloroplast, klorofil ışığı emer.\n3. Tepkime: CO₂+H₂O+Işık→C₆H₁₂O₆+O₂\n4. Işık aşaması: Fotosistem II/I, elektron taşıma.\n5. Calvin döngü: Karbon fiksasyonu, glikoz.\n6. Önem: Oksijen, besin zinciri, karbon döngüsü."),
    ("DNA nedir? Adım adım açıkla.",
     "Adım adım DNA:\n1. Ad: Deoksiribonükleik Asit\n2. Yapı: Çift sarmal, nükleotidler.\n3. Bazlar: A-T, G-C eşleşir.\n4. Fonksiyon: Genetik bilgi saklar.\n5. Replikasyon: DNA polimeraz ile çoğalır.\n6. Transkripsiyon: DNA→mRNA\n7. Translasyon: mRNA→Protein"),
    ("I. Dünya Savaşı nedenleri ve sonuçları? Adım adım analiz.",
     "Adım adım analiz:\nNedenler:\n1. Emperyalizm: Sömürge yarışı\n2. Milliyetçilik: Balkan gerilimleri\n3. Militarizm: Silahlanma\n4. İttifaklar: Üçlü İttifak vs İtilaf\n5. Tetik: Avusturya veliahtı suikaste\nSonuçlar:\n1. 17M ölüm\n2. İmparatorluklar yıkıldı\n3. Versay Antlaşması\n4. II. DS zemini hazırlandı."),
    ("II. Dünya Savaşı nedenleri ve sonuçları? Adım adım analiz.",
     "Adım adım analiz:\nNedenler:\n1. Versay: Almanya'yı ağır cezalandırdı\n2. 1929 Buhranı: Ekonomik çöküş\n3. Faşizm: Hitler, Mussolini\n4. Polonya işgali (1 Eylül 1939)\nSonuçlar:\n1. 70-85M ölüm\n2. Holokost\n3. Atom bombası\n4. Soğuk Savaş\n5. BM kuruldu"),
    ("Kuantum bilgisayar nedir? Adım adım açıkla.",
     "Adım adım kuantum bilgisayar:\n1. Bit vs Qubit: 0/1 vs 0 VE 1 aynı anda\n2. Süperpozisyon: Qubit birden fazla durumda\n3. Dolanıklık: Qubitler birbirine bağlı\n4. Kuantum kapıları: Hadamard, CNOT\n5. Üstünlük: Shor (kripto), Grover (arama)\n6. Zorluk: Decoherence, hata düzeltme\n7. Uygulama: Kripto, ilaç, AI"),
    ("Blockchain nedir? Adım adım açıkla.",
     "Adım adım blockchain:\n1. Tanım: Dağıtık, değiştirilemez veri defteri\n2. Yapı: Blok=veri+hash+önceki hash\n3. Hash: SHA-256 ile bağlanır\n4. Konsensüs: PoW (madencilik) veya PoS (stake)\n5. Değiştirilemezlik: Bir bloğu değiştirmek=tüm zinciri değiştirir\n6. Uygulama: Kripto, NFT, akıllı kontrat"),
    ("Yapay zeka nedir? Adım adım açıkla.",
     "Adım adım yapay zeka:\n1. Tanım: Bilgisayarların insan benzeri zeka göstermesi\n2. Kategoriler: Dar AZ, Genel AZ, Süper AZ\n3. Alt dallar: ML, derin öğrenme, NLP, bilgisayar görüsü\n4. Eğitim: Veri→Model→Eğitim→Değerlendirme\n5. Transformer: Attention, BERT, GPT\n6. Güncel: LLM'ler, çoklu modaller"),
    ("Evrim teorisi nedir? Adım adım açıkla.",
     "Adım adım evrim:\n1. Darwin: Doğal seçilimle türler değişir\n2. Değişkenlik: Mutasyonlar\n3. Kalıtım: DNA ile aktarılır\n4. Doğal seçilim: Uygun bireyler hayatta kalır\n5. Türleşme: İzolasyon→yeni tür\n6. Kanıtlar: Fosil, anatomi, DNA\n7. Önem: Biyolojinin temel teorisi"),
    ("İklim değişikliği nedir? Adım adım analiz.",
     "Adım adım iklim değişikliği:\n1. Tanım: Sera gazları→küresel ısınma\n2. Nedenler: Fosil yakıt, ormansızlaşma, sanayi\n3. Gazlar: CO₂, metan, N₂O\n4. Sonuçlar: Buzul erimesi, deniz yükselmesi, aşırı hava\n5. Önlemler: Yenilenebilir enerji, Paris Anlaşması"),
    ("Bitcoin nasıl çalışır? Adım adım açıkla.",
     "Adım adım Bitcoin:\n1. Tanım: Merkeziyetsiz dijital para\n2. Blockchain: Dağıtık veritabanı\n3. Madencilik: SHA-256 bulmaca\n4. Cüzdan: Public/Private key\n5. İşlem: İmza→Ağ→Madenci→Blok\n6. Sınır: 21M BTC, 4 yılda yarılanma"),
]
for q, a in aca_qa * 1000:
    add_sample(q, a)
print('  Akademik CoT: 10000')

# Derin Düşünme (5000)
deep_qa = [
    ("Bir problemi çözerken hangi adımları izlersin? Düşünce sürecini açıkla.",
     "Derin düşünme sürecim:\n1. Problemi anlama: Ne istiyor? Kısıtlamalar?\n2. Bölme: Büyük problemi küçük parçalara ayırma\n3. Bilgi toplama: İlgili formüller, kavramlar\n4. Plan: Çözüm yolunu oluşturma\n5. Adım adım çözüm: Her adımı dikkatle uygulama\n6. Kontrol: Sonuç mantıklı mı?\n7. Özet: Net sunum\nÖz-değerlendirme: 'Başka yaklaşım daha iyi mi?'"),
    ("Karmaşık karar verirken nasıl düşünsün? Örnekle açıkla.",
     "Karmaşık karar süreci:\nSenaryo: İş teklifi vs girişim\n1. Durum analizi: Mevcut durum, hedefler\n2. Faktörler: Gelir, öğrenme, risk, yaşam tarzı\n3. Artı/Eksi: İş=güvenli ama sınırlı, girişim=riskli ama yüksek potansiyel\n4. Varsayımları sorgulama\n5. Karar: Risk toleransına göre değişir\nÖz-değerlendirme: '5 yıl sonra ne olacak?'"),
    ("Argüman analizi nasıl yapılır? Adım adım açıkla.",
     "Argüman analizi:\n1. Argümanı belirle: Önerme+kanıtlar\n2. Mantıksal yapı: Dedüktif veya endüktif?\n3. Kanıt kalitesi: Güvenilir kaynak? Yeterli örnek?\n4. Mantıksal hatalar: Ad homine, korkutma, yanlış ikilem\n5. Karşı argüman: Karşıt görüşün güçlü yönleri?\n6. Sonuç: Güçlü mü zayıf mı?\nÖnemli: Tarafsız kalmak, önyargıları sorgulamak."),
    ("Etik ikilem nasıl çözülür? Örnek üzerinden açıkla.",
     "Etik ikilem çözümü:\nSenaryo: Otonom araç kime zarar vermeli?\n1. Paydaşlar: Sürücü, yaya, diğer araçlar\n2. Etik çerçeveler:\n   - Sonuççuluk: En az zarar\n   - Kuralla ilgi: Evrensel kural?\n   - Erdem: Adil ve dürüst ne yapar?\n3. Karar: Can kaybını en aza indiren seçenek\nÖz-değerlendirme: 'Kendi ailem için kabul eder miydim?'\nAltın kural: Diğerinin yerinde olsam ne isterdim?"),
    ("Yanlış inanç nasıl düzeltilir? Adım adım açıkla.",
     "Yanlış İnanç Düzeltme:\n1. Farkındalık: Yanlış olduğunu kabul et\n2. Kanıt ara: Güvenilir kaynaklardan veri topla\n3. Karşıt kanıtlar: Çürüten kanıtları da ara\n4. Mantıksal çıkarım: 'A doğru, B doğru, C doğru mu?'\n5. Uzman görüşü: Uzmanlara danış\n6. Test et: Yeni anlayışı uygula\n7. Geri bildirim: Sonuçları değerlendir\nÖnemli: Aceleci sonuç çıkarmak en büyük düşman."),
]
for q, a in deep_qa * 1000:
    add_sample(q, a)
print('  Derin Düşünme: 5000')

# Kültür/Sohbet (6000)
cult_qa = [
    ("Türkiye'nin başkenti neresidir? Detaylı açıkla.",
     "Türkiye'nin başkenti Ankara'dır. 13 Ekim 1923'te başkent ilan edilmiştir. Nüfusu yaklaşık 6 milyon olan Ankara, Türkiye'nin ikinci en büyük şehridir."),
    ("Atatürk kimdir? Detaylı açıkla.",
     "Mustafa Kemal Atatürk (1881-1938), Türkiye Cumhuriyeti'nin kurucusu ve ilk Cumhurbaşkanıdır. Kurtuluş Savaşı'nı önderleyerek bağımsız Türkiye'yi kurmuş, çağdaşlaşma reformları yapmıştır."),
    ("Osmanlı İmparatorluğu ne zaman kuruldu? Detaylı açıkla.",
     "Osmanlı İmparatorluğu 1299'da Osman Bey tarafından kurulmuştur. 1453'te İstanbul'un fethiyle büyük güce ulaşmış, 1922'de sona ermiştir. Toplam 623 yıl sürmüştür."),
    ("Türk mutfağının önemli lezzetleri nelerdir? Adım adım açıkla.",
     "Türk mutfağı lezzetleri:\n1. Ana Yemekler: Kebap çeşitleri (Adana, Urfa, İskender), köfte, karnıyarık, mantı, lahmacun.\n2. Tatlılar: Baklava, künefe, katmer, lokum, helva, sütlaç.\n3. İçecekler: Türk kahvesi, çay, ayran.\n4. Kahvaltı: Simit, peynir, zeytin, bal, kaymak."),
    ("Türkiye'nin UNESCO Dünya Mirası listesindeki yerleri nelerdir?",
     "Türkiye UNESCO Mirasları:\n1. Kapadokya (Nevşehir)\n2. Pamukkale (Denizli)\n3. Efes (İzmir)\n4. Nemrut Dağı (Adıyaman)\n5. Safranbolu (Karabük)\n6. İstanbul Tarihi Alanlar\n7. Göbeklitepe (Şanlıurfa)\n8. Sümela Manastırı (Trabzon)"),
]
for q, a in cult_qa * 1200:
    add_sample(q, a)
print('  Kültür/Sohbet: 6000')

# Sohbet (1000)
chat_qa = [
    ("Merhaba! Kendini tanıtır mısın?", "Merhaba! Ben KuroNeko TR, Türkçe konuşan bir yapay zeka asistanıyım. Size yardımcı olmak için buradayım."),
    ("Teşekkürler!", "Rica ederim! Başka sorunuz varsa yardımcı olmaktan mutluluk duyarım."),
    ("Görüşürüz!", "Görüşmek üzere! İyi günler dilerim."),
    ("Yardım edebilir misin?", "Tabii ki! Ne konuda yardıma ihtiyacınız var?"),
    ("Sen kimsin?", "Ben KuroNeko TR, Türkçe konuşan zeki bir yapay zeka asistanıyım."),
    ("Ne yapabilirsin?", "Birçok şey yapabilirim: Sorularınıza yanıt verebilir, kod yazabilir, metin oluşturabilir, çeviri yapabilir, matematik çözebilir, sohbet edebilirim."),
]
for q, a in chat_qa * 167:
    add_sample(q, a)
print('  Sohbet: ~1000')

# Shuffle ve split
random.shuffle(all_data)
val_size = int(len(all_data) * 0.05)
val_data = all_data[:val_size]
train_data = all_data[val_size:]

with open('/kaggle/working/data/train.jsonl', 'w', encoding='utf-8') as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
with open('/kaggle/working/data/val.jsonl', 'w', encoding='utf-8') as f:
    for item in val_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print('\n' + '='*50)
print('✅ VERİ SETİ HAZIR')
print('Train: ' + str(len(train_data)) + ' | Val: ' + str(len(val_data)) + ' | Toplam: ' + str(len(all_data)))

In [ ]:
# ── 5. MODEL YÜKLE (QLoRA 4-bit) ────────────────────────────────────────────
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print('📥 Model yükleniyor...')
t0 = time.time()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model_name = 'microsoft/Phi-4-mini-instruct'

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, token=HF_TOKEN or None)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    token=HF_TOKEN or None,
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ Model hazır! (' + str(int(time.time()-t0)) + 's)')

In [ ]:
# ── 6. EĞİTİM ───────────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

print('🚀 Eğitim başlıyor...')
t0 = time.time()

train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)

def fmt(examples):
    texts = []
    for i in range(len(examples['q'])):
        q, a = examples['q'][i], examples['a'][i]
        texts.append('<|im_start|>user\n' + q + '<|im_end|>\n<|im_start|>assistant\n' + a + '<|im_end|>')
    return {'text': texts}

train_ds = train_ds.map(fmt, batched=True, batch_size=2000)
val_ds = val_ds.map(fmt, batched=True, batch_size=2000)

gpu_count = torch.cuda.device_count()
batch_size = 2 if gpu_count >= 2 else 1
grad_accum = 8

print('GPU: ' + str(gpu_count) + 'x | batch=' + str(batch_size) + ' | grad_accum=' + str(grad_accum))
print('Train: ' + str(len(train_ds)) + ' | Val: ' + str(len(val_ds)))

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    dataset_text_field='text', max_seq_length=2048,
    args=TrainingArguments(
        output_dir='/kaggle/working/output',
        num_train_epochs=3,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=2e-4, warmup_steps=100,
        logging_steps=50, save_steps=500,
        optim='adamw_8bit', lr_scheduler_type='cosine',
        report_to='none', seed=42, bf16=True,
        evaluation_strategy='steps', eval_steps=500,
        load_best_model_at_end=True,
    ),
)

result = trainer.train()
elapsed = (time.time()-t0)/60
print('\n✅ Eğitim bitti! ' + str(round(elapsed,1)) + ' dk | Loss: ' + str(round(result.training_loss,4)))

In [ ]:
# ── 7. TEST ──────────────────────────────────────────────────────────────────
print('🧪 MODEL TESTİ')
model.eval()
tests = [
    'Merhaba! Kendini tanıtır mısın?',
    'Türkiye\'nin başkenti neresidir?',
    'Python\'da liste nasıl sıralanır? Adım adım açıkla.',
    'Fotosentez nedir? Adım adım açıkla.',
    '15 TL/kg elma, 20 TL/kg armut. 3kg elma + 2kg armut = ? Adım adım çöz.',
    'Docker container arka planda nasıl çalıştırılır?',
    'Kuantum bilgisayar nedir? Adım adım açıkla.',
    'Bir problemi çözerken hangi adımları izlersin?',
    'İklim değişikliği nedir? Adım adım analiz et.',
    'Git\'te branch oluştur, commit, merge adımları?',
]
for q in tests:
    prompt = '<|im_start|>user\n' + q + '<|im_end|>\n<|im_start|>assistant\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True, top_p=0.9, repetition_penalty=1.1)
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print('\n❓ ' + q)
    print('🤖 ' + resp.strip())
    print('-'*50)

In [ ]:
# ── 8. PUSH TO HUB ──────────────────────────────────────────────────────────
HUB_ID = 'KuroNeko1234t/phi4-mini-tr'
if HF_TOKEN:
    print('📤 Push ediliyor...')
    merged = model.merge_and_unload()
    merged.save_pretrained('/kaggle/working/merged')
    tokenizer.save_pretrained('/kaggle/working/merged')
    from huggingface_hub import create_repo
    create_repo(HUB_ID, token=HF_TOKEN, exist_ok=True, repo_type='model')
    merged.push_to_hub(HUB_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HUB_ID, token=HF_TOKEN)
    print('✅ https://huggingface.co/' + HUB_ID)
else:
    print('⚠️ HF_TOKEN yok, push atlandı')
print('🎉 TAMAM!')